# ANN Dynamic Pricing Optimization System

This notebook is a portfolio walkthrough for the leakage-safe version of the project. The deployed pipeline predicts demand at proposed prices and then selects a constrained price scenario. It does **not** use realized demand or an engineered optimal-price label as live inputs.


## 1. Setup

Run this notebook from the project root. The committed model artifacts allow inference without retraining.


In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "torch")

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

from src.constants import DEFAULT_RECORD
from src.data_generation import generate_synthetic_pricing_data
from src.prediction_pipeline import DynamicPricingPipeline


## 2. Generate or inspect reproducible data

The full dataset is synthetic and generated locally. This avoids presenting fabricated observations as real commercial data.


In [ ]:
demo_data = generate_synthetic_pricing_data(2_000)
demo_data.head()


In [ ]:
demo_data[["current_price", "competitor_price", "realized_demand", "revenue", "gross_margin"]].describe().T


## 3. Load validated model metrics


In [ ]:
metrics = json.loads((PROJECT_ROOT / "outputs" / "model_metrics.json").read_text())
metrics


## 4. Load the production-style pipeline


In [ ]:
pipeline = DynamicPricingPipeline(PROJECT_ROOT / "models")


## 5. Forecast and optimize one product


In [ ]:
result = pipeline.recommend_one(DEFAULT_RECORD, objective_label="Maximize Margin", n_grid=31)
{k: v for k, v in result.items() if k not in {"scenario_table", "rule_notes"}}


## 6. Inspect price scenarios


In [ ]:
scenario_table = result["scenario_table"]
scenario_table[["current_price", "predicted_demand", "predicted_revenue", "predicted_margin", "selected"]].head(10)


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(scenario_table["current_price"], scenario_table["predicted_demand"], marker="o")
ax1.set_xlabel("Candidate price")
ax1.set_ylabel("Predicted demand")
ax2 = ax1.twinx()
ax2.plot(scenario_table["current_price"], scenario_table["predicted_margin"], linestyle="--")
ax2.set_ylabel("Predicted gross margin")
ax1.axvline(result["recommended_price"], linestyle=":")
plt.title("Price scenario optimization")
plt.show()


## 7. Batch optimization


In [ ]:
sample = pd.read_csv(PROJECT_ROOT / "data" / "sample_input.csv")
batch_result = pipeline.recommend_batch(sample, objective_label="Balance Demand and Margin", n_grid=13)
batch_result[["product_id", "current_price", "recommended_price", "pricing_action", "estimated_margin_uplift"]]


## 8. Optional retraining

From a terminal in the project root:

```bash
pip install -r requirements-training.txt
python -m src.train --samples 16000 --epochs-regression 20 --epochs-classification 5
```

The training stage uses the JAX Keras backend; the saved `.keras` artifacts are then evaluated and deployed with the PyTorch Keras backend.


## 9. Interpretation and limitations

- MAE communicates average unit error; RMSE emphasizes larger misses; R² measures explained variation; MAPE provides a relative error view.
- Scenario curves explain how the ANN responds to proposed prices.
- The optimization is predictive rather than causal. Real pricing decisions require experiments, approvals, drift monitoring, and consumer/legal review.
- Synthetic validation demonstrates engineering quality, not expected financial impact on a real business.
